In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import lxml
from selenium import webdriver
import time


In [33]:
# %pip install lxml
# %pip install selenium

In [5]:
urls = {
    "US": "https://www.ysl.com/en-us/ca/shop-women/handbags",
    "FR": "https://www.ysl.com/en-fr/shop-women/handbags",
    "IT": "https://www.ysl.com/en-it/shop-women/handbags",
    "CN": "https://www.ysl.cn/categories/shop-women/handbags/view-all.html"
}

driver = webdriver.Chrome() # use selenium for javascript -- prices won't load with just soup

In [6]:
def scroll_to_bottom(driver):

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:

        # scroll in small steps
        for i in range(10):
            driver.execute_script("window.scrollBy(0, 600)")
            time.sleep(0.5)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            break

        last_height = new_height

In [7]:
data = []

for country, url in urls.items():
    driver.get(url)
    time.sleep(5)

    scroll_to_bottom(driver) # to get all the bags

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # china html is different
    if country == "CN":
        titles = soup.select(".productName")
        prices = soup.select(".productPrice")
    else:
        titles = soup.select('h2[data-qa^="plp-product-title"]')
        prices = soup.select('span[data-qa^="plp-product-price"]')

    # was for debugging
    # print("titles:", len(titles))
    # print("prices:", len(prices))

    for t,p in zip(titles, prices):
        name = t.get_text(strip=True)
        priceText = p.get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", priceText))

        if country == "CN": # china product ID is also stored differently - in the product URL only
            link = t.find_parent("a")["href"]
            sku = link.split("/")[-1].split("?")[0].replace(".html", "").upper()
        else:
            sku = t["data-qa"].replace("plp-product-title-", "")

        currency = re.findall(r"[^\d.,\s]", priceText)[0]

        data.append({
        "brand": "Saint Laurent",
        "product_name": name,
        "price": price,
        "currency": currency,
        "country": country,
        "sku": sku
})


In [46]:
df = pd.DataFrame(data)
print(df)
print(len(df))


              brand               product_name    price currency country  \
0     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
1     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
2     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
3     Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
4     Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
...             ...                        ...      ...      ...     ...   
1463  Saint Laurent          KATE流苏中号鳄鱼纹压纹皮革手袋  22500.0        ¥      CN   
1464  Saint Laurent          KATE流苏中号鳄鱼纹压纹皮革手袋  22500.0        ¥      CN   
1465  Saint Laurent             KATE流苏小号粒面皮革手袋  18500.0        ¥      CN   
1466  Saint Laurent          KATE流苏小号鳄鱼纹压纹皮革手袋  19900.0        ¥      CN   
1467  Saint Laurent          KATE流苏小号鳄鱼纹压纹皮革手袋  19900.0        ¥      CN   

                  sku  
0     851432AAGWJ1000  
1     851432AAGWK2050  
2     851432AAG

In [47]:
df_cleaned = df.drop_duplicates(subset=['product_name', 'price', 'country'], keep='first') # some bags are listed multiple times with the same name and price - SKU differs because the bag is a diff color. drop the duplicates because we only care about the model.


In [48]:
df_cleaned.head(10)

,brand,product_name,price,currency,country,sku
0,Saint Laurent,MOMBASA small in leather,3450.0,$,US,851432AAGWJ1000
3,Saint Laurent,MOMBASA medium in leather,4300.0,$,US,862029AAGWJ1000
6,Saint Laurent,MOMBASA large in leather,5600.0,$,US,A0011MAAGWJ1000
8,Saint Laurent,MOMBASA large in pony hair leather,6100.0,$,US,848811AAFPH1042
9,Saint Laurent,MOMBASA medium in vintage leather,4500.0,$,US,862029AAFDJ2551
10,Saint Laurent,MOMBASA medium in pony hair leather,4500.0,$,US,862029AAGE22077
11,Saint Laurent,MOMBASA small in alligator,25000.0,$,US,850571EABF53322
12,Saint Laurent,ICARINO mini in quilted nappa,2900.0,$,US,871329AAANG1000
15,Saint Laurent,ICARINO in quilted nappa,3100.0,$,US,851689AAANG1000
18,Saint Laurent,ICARINO in quilted suede,3100.0,$,US,8516891U8072916


In [49]:
print(len(df_cleaned))

811


In [29]:
df_cleaned.to_csv("data/ysl_allbags_cleaned.csv", index=False)

In [50]:
pivot_df = df_cleaned.pivot_table(index="sku", columns="country", values="price", aggfunc="first")

num_all_4 = (pivot_df.notna().sum(axis=1) == 4).sum()
print(num_all_4) # num of bags (check by SKU) present in all 4 countries, minus duplicates

82


In [51]:
all4 = pivot_df[pivot_df.notna().all(axis=1)]
all4

country,CN,FR,IT,US
sku,,,,
364021BOW0J1000,18500.0,1950.0,1950.0,2450.0
37829902G9W1000,28000.0,2950.0,2950.0,3600.0
378299B681N1000,26000.0,2800.0,2800.0,3400.0
39203502G9W2536,20500.0,2200.0,2200.0,2700.0
42186302G9W1000,23500.0,2500.0,2500.0,2950.0
...,...,...,...,...
862558AACX71000,21500.0,2200.0,2200.0,2700.0
8627121U8P72916,24000.0,2600.0,2600.0,3200.0
862712AAB321000,23000.0,2500.0,2500.0,3000.0


In [52]:
# randomly select 10 bags

sample_skus = all4.sample(10, random_state=42).index
df_40 = df[df["sku"].isin(sample_skus)].copy()
df_40 = df_40.reset_index(drop=True)
df_40


,brand,product_name,price,currency,country,sku
0,Saint Laurent,JAMIE SHOPPING SMALL IN LAMBSKIN,3800.0,$,US,833948AAB321000
1,Saint Laurent,LE 5 À 7 supple LARGE in grained leather,3200.0,$,US,753837AAAUQ1000
2,Saint Laurent,NIKI medium in vintage leather,3200.0,$,US,6331840EN076195
3,Saint Laurent,MINI LE 5 À 7 IN SMOOTH LEATHER,1900.0,$,US,7103182R20W1000
4,Saint Laurent,SAC DE JOUR IN SUPPLE GRAINED LEATHER - BABY,3200.0,$,US,717448DTI0W1000
5,Saint Laurent,SAC DE JOUR IN SMOOTH LEATHER - BABY,2950.0,$,US,42186302G9W1000
6,Saint Laurent,LE 37 small in shiny leather,2900.0,$,US,7490362R20W1000
7,Saint Laurent,KATE MEDIUM IN GRAIN DE POUDRE EMBOSSED LEATHER,2450.0,$,US,364021BOW0J1000
8,Saint Laurent,LARGE Jamie 4.3 in lambskin,4600.0,$,US,742431AAB321000
9,Saint Laurent,SHOPPING SAINT LAURENT IN LEATHER,1550.0,$,US,600306CSV0J1000


In [53]:
df_40[df_40["country"] == "US"]["sku"].is_unique

True

In [54]:
# each SKU repeats 4 times (each country), but the names differ bc of language. translate all the names to eng:

us_map = df_40[df_40["country"] == "US"].set_index("sku")["product_name"]
df_40["product_name"] = df_40["sku"].map(us_map)

In [55]:
df_40

,brand,product_name,price,currency,country,sku
0,Saint Laurent,JAMIE SHOPPING SMALL IN LAMBSKIN,3800.0,$,US,833948AAB321000
1,Saint Laurent,LE 5 À 7 supple LARGE in grained leather,3200.0,$,US,753837AAAUQ1000
2,Saint Laurent,NIKI medium in vintage leather,3200.0,$,US,6331840EN076195
3,Saint Laurent,MINI LE 5 À 7 IN SMOOTH LEATHER,1900.0,$,US,7103182R20W1000
4,Saint Laurent,SAC DE JOUR IN SUPPLE GRAINED LEATHER - BABY,3200.0,$,US,717448DTI0W1000
5,Saint Laurent,SAC DE JOUR IN SMOOTH LEATHER - BABY,2950.0,$,US,42186302G9W1000
6,Saint Laurent,LE 37 small in shiny leather,2900.0,$,US,7490362R20W1000
7,Saint Laurent,KATE MEDIUM IN GRAIN DE POUDRE EMBOSSED LEATHER,2450.0,$,US,364021BOW0J1000
8,Saint Laurent,LARGE Jamie 4.3 in lambskin,4600.0,$,US,742431AAB321000
9,Saint Laurent,SHOPPING SAINT LAURENT IN LEATHER,1550.0,$,US,600306CSV0J1000


In [56]:
#check:
subset = df_40[df_40["product_name"] == "LARGE Jamie 4.3 in lambskin"]
subset

,brand,product_name,price,currency,country,sku
8,Saint Laurent,LARGE Jamie 4.3 in lambskin,4600.0,$,US,742431AAB321000
18,Saint Laurent,LARGE Jamie 4.3 in lambskin,3700.0,€,FR,742431AAB321000
28,Saint Laurent,LARGE Jamie 4.3 in lambskin,3700.0,€,IT,742431AAB321000
32,Saint Laurent,LARGE Jamie 4.3 in lambskin,35000.0,¥,CN,742431AAB321000


In [57]:
counts = df_40["product_name"].value_counts()
print(counts)

product_name
JAMIE SHOPPING SMALL IN LAMBSKIN                   4
LE 5 À 7 supple LARGE in grained leather           4
NIKI medium in vintage leather                     4
MINI LE 5 À 7 IN SMOOTH LEATHER                    4
SAC DE JOUR IN SUPPLE GRAINED LEATHER - BABY       4
SAC DE JOUR IN SMOOTH LEATHER - BABY               4
LE 37 small in shiny leather                       4
KATE MEDIUM IN GRAIN DE POUDRE EMBOSSED LEATHER    4
LARGE Jamie 4.3 in lambskin                        4
SHOPPING SAINT LAURENT IN LEATHER                  4
Name: count, dtype: int64


In [58]:
df_40.to_csv("data/ysl_10bags_4countries_cleaned.csv", index=False)